In [3]:
from google import genai
from google.genai import types
from PIL import Image

In [4]:
# 1. Initialize the client (automatically picks up GEMINI_API_KEY from environment)
client = genai.Client(
    vertexai=True,
    project="infinitas-ds-ai-poc",
    location="us-central1",
)

In [5]:
# 2. Open your local image file

paths = ["test_images/First_Tamper.png", "test_images/first-tamper.png", "test_images/combined.png",
         "test_images/neat.png", "test_images/neatest.png", "test_images/cropped-obvious.png"]
images = []

for path in paths:
    images.append(Image.open(path))

print(len(images))

6


In [6]:
# Read the V1 prompt

with open("prompt-library/V1.txt", "r", encoding="utf-8") as file:
    file_content = file.read()

prompt_v1 = file_content
print(type(prompt_v1))

# Read the V1 prompt split into system instruction + task prompt

with open("prompt-library/V1_system.txt", "r", encoding="utf-8") as file:
    system_prompt_v1 = file.read()

with open("prompt-library/V1_task.txt", "r", encoding="utf-8") as file:
    task_prompt_v1 = file.read()

<class 'str'>


## Try the first two image together

In [7]:
image_list = images[0:2] # image 1 and 2

for image in image_list:

    # 3. Generate content by passing both the image object and your text prompt
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            prompt_v1,
            image
        ], config = types.GenerateContentConfig(
            temperature = 0.1
        )
    )

    # 4. Print the text result
    print(response.text)

```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "The document was thoroughly scanned for visual inconsistencies indicative of post-recapture digital alteration. All text, including the title, address lines, and date, exhibits a uniform font style, weight, and color (black on white). The sharpness and resolution are consistent across the entire image, with no areas appearing blurrier or sharper than others. There are no detectable anomalies in shadows, edges, or background around any specific characters or text blocks. The spacing and baseline alignment are also consistent throughout. The slight difference in font size for the title is a standard formatting choice and not an indication of tampering. Overall, the visual evidence strongly suggests that the document is a single, unaltered digital generation.",
  "signals_detected": [],
  "notes": "The document is synthetic and digitally generated, as stated in the prompt. The analysis confirms its visual integrit

In [8]:
# V1 split: persona passed as system_instruction, task prompt as the user turn

image_list = images[0:2] # image 1 and 2

for image in image_list:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[task_prompt_v1, image],
        config=types.GenerateContentConfig(
            system_instruction=system_prompt_v1,
            temperature = 0.1
        ),
    )
    print(response.text)

```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "The document exhibits a consistent typography baseline across all text regions. The font style, weight, sharpness, and resolution are uniform throughout. There are no discernible font size mismatches, mismatched shadows, irregular edges, or color/contrast discontinuities around any specific text areas. All characters, including numbers, appear typographically consistent with the surrounding prose. The overall appearance suggests a clean, digitally generated document without any post-recapture alterations.",
  "signals_detected": [],
  "notes": ""
}
```
```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "The document displays consistent typography throughout. All text regions, including the title, address lines, and date, exhibit uniform font weight, sharpness, and resolution. There are no discernible font size mismatches, color or contrast discontinuities, or anomalies in shadows or edge

### Try with other images
- separated for ease of reading

In [9]:
# Other images
image_list = images[2:7] #image 3,4,5,6

for image in image_list:

    # 3. Generate content by passing both the image object and your text prompt
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            prompt_v1,
            image
        ], config = types.GenerateContentConfig(
            temperature = 0.1)
    )

    # 4. Print the text result
    print(response.text)

```json
{
  "verdict": "tampered",
  "confidence": "high",
  "reasoning": "The document exhibits clear visual signals of post-recapture digital alteration in two distinct areas. Both the date field ('พฤศจิกายน ๒๕๖๙') and the amount field ('๖๐๐,๐๐๐') are enclosed within digitally inserted gray rectangular backgrounds. These backgrounds are not consistent with the rest of the document's visual style and indicate an overlay or modification. Furthermore, the text within these gray boxes appears to have a different sharpness, resolution, and possibly font weight compared to the adjacent, untampered text, reinforcing the conclusion that these specific text elements were digitally altered or inserted.",
  "signals_detected": [
    {
      "signal_type": "color_contrast_discontinuity",
      "location": "The gray rectangular background around the date 'พฤศจิกายน ๒๕๖๙'.",
      "severity": "high"
    },
    {
      "signal_type": "resolution_mismatch",
      "location": "The text 'พฤศจิกายน ๒๕๖

In [10]:
# Other images with separate prompts
image_list = images[2:7] #image 3,4,5,6

for image in image_list:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[task_prompt_v1, image],
        config=types.GenerateContentConfig(
            system_instruction=system_prompt_v1,
            temperature = 0.1
        ),
    )
    print(response.text)

```json
{
  "verdict": "tampered",
  "confidence": "high",
  "reasoning": "The document exhibits clear visual signals of digital alteration. Specifically, two distinct text regions, the date and the loan amount, are enclosed within rectangular gray boxes. The text within these gray boxes ('๙ พฤศจิกายน ๒๕๖๙' for the date and '๖๐๐,๐๐๐' for the amount) shows inconsistencies in font rendering, sharpness, and potentially font style when compared to the surrounding, unboxed text. The presence of these artificially colored background boxes, along with the differing text characteristics within them, strongly indicates that these specific pieces of information were digitally inserted or modified after the document was initially captured.",
  "signals_detected": [
    {
      "signal_type": "color_contrast_discontinuity",
      "location": "Date field: '๙ พฤศจิกายน ๒๕๖๙'",
      "severity": "high"
    },
    {
      "signal_type": "font_weight_inconsistency",
      "location": "Date field: '๙ พฤ

In [11]:
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      avg_logprobs=-1.2320455575918223,
      content=Content(
        parts=[
          Part(
            text="""```json
{
  "verdict": "authentic",
  "confidence": "high",
  "reasoning": "The document's typography, including font weight, sharpness, resolution, and color, is highly consistent across all text regions. Specifically, the numerical value '๖๐๐,๐๐๐' (600,000) matches the surrounding Thai text in every visual aspect. There are no signs of mismatched shadows, edges, backgrounds, or color/contrast discontinuities around any specific characters or numbers. All text appears to be rendered uniformly, suggesting it is part of the original digital generation rather than a post-recapture alteration.",
  "signals_detected": [],
  "notes": ""
}
```"""
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>
    ),
  ],
  create_time=datetime.dat